Seguindo a mesma estrutura do KNN, Vamos começar com o problema:

Imagine que você acaba de receber um email. O sistema precisa decidir se essa mensagem que acabou de chegar, com o assunto "Esse está pagando! Aposte agora", é um email importante ou se é spam. Esse é o problema de classificação classico de **Naive Bayes**.

Vamos construir isso:

**Probabilidade a Priori**

A probabilidade a *Priori* é a nossa crença sobre algo antes de analisar qualquer nova evidência. Por exemplo:

- O algoritmo olha para todo o seu histórico de e-mails e calcula a proporção de cada categoria.

- Se de 10.000 emails recebidos, 2.000 foram Spam e 8.000 não eram, as probabilidades a priori são:
    - $P(Spam) = \frac{2.000}{10.000} = 20\% \Rightarrow$
    - $P(Não\space Spam) = 80\%$
    
- Isso significa que, sem ler o novo email, já sabemos que há uma chance de 20% de ser spam.

**Verossimilhança**

A verossimilhança (*likelihood*) mede o quão provável é encontrar a evidência (as palavras do email) assumindo que ele pertence a uma determinada categoria. Por exemplo:

- Se este email fosse um Spam, qual seria a probabilidade de ele conter as palavras "Pagando", "Aposte", "agora"? e se não fosse um Spam?

- O algoritmo volta ao histórico e calcula a frequência das palavras dentro de cada categoria:
    - **Spams**
        - $P("Pagando"|Spam) = 40\%$ (40% de todos os spams continham a palavra "Pagando")
        - $P("Aposte"|Spam) = 30\%$
    
    - **Não Spams**
        - $P("Pagando|Não \space Spam) = 1\%$ 
        - $P("Aposte"|Não \space Spam) = 0.5\%$

Nota: Como retomaremos o Naive Bayes assume que a presença de uma palavra é independente da outra. Isso significa que podemos simplesmente multiplicar as probabilidades individuais:

- Verossimilhança (Spam):

    - $P(\text{Evidência}|Spam) = P("Pagando"|Spam) \times P("Aposte"|Spam) ...$

- Verossimilhança (Não Spam):

    - $P(\text{Evidência}|Não \space Spam) = P("Pagando"|Não \space Spam) \times P("Aposte"|Não \space Spam) ...$

**Probabilidade a Posteriori**

A probabilidade a *posteriori* é o resultado final que o algoritmo nos entrega. É a probabilidade de um e-mail pertencer a uma categoria, depois de termos analisado a evidência (das palavras). A fórmula de Bayes nos diz como faz isso:

$P(y|X) = \frac{P(X|y) \times P(y)}{P(X)}$

Onde:

- $P(y|X):$ Probabilidade a *posteriori*, Probabilidade da categoria (classe) $y$ (spam ou não spam) dado evidência (features) $X$
- $P(X|y):$ Verossimilhança das features $X$ dado classe $y$
- $P(y):$ Probabilidade a *priori* da classe $y$
- $P(X):$ Evidência

**Classificação**

Finalmente, note que apesar da probabilidade a priori favorecer "Não Spam" (80:20), a evidência contida nas palavras gera uma verossimilhança tão maior para a categoria "Spam" que, no final, a probabilidade a posteriori de ser Spam será a mais alta!

### 3.2.2 Intuição 

Vamos ver isso na prática, dessa vez usaremos o SMS Spam Collection Dataset:

Você recebe um SMS com o texto:

- WINNER!! You have won a £1000 prize! Call now to claim!

Pergunta: **Você classificaria esse email como `spam` ou `ham` (não spam)?**

In [26]:
# =======================================================
# IMPORTANDO AS BIBLIOTECAS
# =======================================================




In [27]:
# Carregando o Dataset
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "uciml/sms-spam-collection-dataset",
  "spam.csv",
  pandas_kwargs={"encoding": "latin", "usecols": [0,1], "names": ["class", "sms"], 'header': 0}
)
# =================================================================


#### **Desafio: Pré-processamento**

Dessa Vez, precisamos processar o texto:

Ideia: 

1. Transformar todo o texto em minúsculo

2. Remover pontuações

3. Tokenizar (separar as palavras)

4. Remover stopwords (palavras comuns como "e", "o") - opcional, mas boa prática

Faça isso!

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

In [29]:
# IGNORE ================================================
from build.utils.text import preprocess_text

try: 
    df['words'] = df['sms'].apply(preprocess_text)
except Exception as e:
    print("Não existe a função preprocess_text")
    pass
# =====================================================

In [30]:
# =======================================================
# Passo 1: Calcular as probabilidades a priori

# Fórmula: P(class) = numbers of class's messages / total messages
# =======================================================

total_messages = len(df)
spam_count = len(df[df['class'] == 'spam'])
ham_count = len(df[df['class'] == 'ham'])

prior_spam = spam_count / total_messages
prior_ham = ham_count / total_messages

print(f"Total de Mensagens: {total_messages}")
print(f"\nTotal de Mensagens Spam: {spam_count}")
print(f"\nTotal de Mensagens Ham: {ham_count}")
print(f"\nP(SPAM): {prior_spam*100:.2f}%")
print(f"\nP(HAM): {prior_ham*100:.2f}%")

print(f'''\nInterpretação:
      Antes de analisar o conteúdo de uma mensagem,
        - Há probabilidade de ser SPAM é de aproximadamente {prior_spam*100:.2f}%.
        - Há probabilidade de ser HAM é de aproximadamente {prior_ham*100:.2f}%.
      ''')


Total de Mensagens: 5572

Total de Mensagens Spam: 747

Total de Mensagens Ham: 4825

P(SPAM): 13.41%

P(HAM): 86.59%

Interpretação:
      Antes de analisar o conteúdo de uma mensagem,
        - Há probabilidade de ser SPAM é de aproximadamente 13.41%.
        - Há probabilidade de ser HAM é de aproximadamente 86.59%.
      


In [33]:
# =======================================================
# Passo 2: Calcular Verossimilhança (Likelihood)

# Fórmula: P(word|class) = (count(word in class) + 1) / (total words in class + V)
# Onde V é o tamanho do vocabulário (número de palavras únicas)
# O "+1" é usado para suavização de Laplace (Laplace Smoothing) - evitar probabilidade zero
# =======================================================

from collections import Counter

spam_words = df[df['class'] == 'spam']['words'].sum()
ham_words = df[df['class'] == 'ham']['words'].sum()

# Contar a frequência das palavras em cada classe
spam_word_count = Counter(spam_words)
ham_word_count = Counter(ham_words)

vocabulary = set(spam_word_count.keys()).union(set(ham_word_count.keys()))

total_spam_words = len(spam_words)
total_ham_words = len(ham_words)

print(f'''Estatísticas de Palavras:
      
  - Vocabulário Total (V): {len(vocabulary)} palavras únicas
  - Total de Palavras em Mensagens SPAM: {total_spam_words}
  - Total de Palavras em Mensagens HAM: {total_ham_words}
      ''')

print(f"Palavras mais comuns em SPAM:")

for word, count in spam_word_count.most_common(5):
    print(f"  {word}: {count}")
print(f"\nPalavras mais comuns em HAM:")
for word, count in ham_word_count.most_common(5):
    print(f"  {word}: {count}")

def likelihood(word, cls):
    if cls == 'spam':
        word_count = spam_word_count.get(word, 0)
        return (word_count + 1) / (total_spam_words + len(vocabulary))
    else:
        word_count = ham_word_count.get(word, 0)
        return (word_count + 1) / (total_ham_words + len(vocabulary))

print(f'''\nExemplo de Cálculo de Verossimilhança:
  P("free"|SPAM) = {likelihood("free", "spam")*10:.4f}%
  P("free"|HAM) = {likelihood("free", "ham")*100:.4f}%
      ''')



Estatísticas de Palavras:

  - Vocabulário Total (V): 8512 palavras únicas
  - Total de Palavras em Mensagens SPAM: 14934
  - Total de Palavras em Mensagens HAM: 61423
      
Palavras mais comuns em SPAM:
  to: 686
  call: 350
  you: 287
  your: 263
  free: 219

Palavras mais comuns em HAM:
  you: 1837
  to: 1554
  the: 1119
  and: 848
  in: 813

Exemplo de Cálculo de Verossimilhança:
  P("free"|SPAM) = 0.0938%
  P("free"|HAM) = 0.0858%
      


In [36]:
# =======================================================
# Passo 3: Classificação de uma nova mensagem
# =======================================================

new_message = "WINNER!! You have won a £1000 prize! Call now to claim!"

preprocessed_message = preprocess_text(new_message) 

# Calcular a probabilidade logarítmica para evitar underflow (falaremos mais sobre isso depois)
log_prob_spam = np.log(prior_spam)
log_prob_ham = np.log(prior_ham)

for word in preprocessed_message:
    log_prob_spam += np.log(likelihood(word, 'spam'))
    log_prob_ham += np.log(likelihood(word, 'ham'))

print(f'''\nClassificação da Mensagem:
    Mensagem: "{new_message}"
    Classificação: {"Spam" if log_prob_spam > log_prob_ham else "Ham"}
''')



Classificação da Mensagem:
    Mensagem: "WINNER!! You have won a £1000 prize! Call now to claim!"
    Classificação: Spam



### 3.1.3 Formulação Formal

**Entrada**
1. Conjunto de treinamento de n amostra em pares $D=\{(x^{(1)}, y^{(1)}), (x^{(2)}, x^{(i)}), ..., x^{(n)}, y^{(n)}\}$

    onde:

    - $x^{(i)}$: vetor de d features ($x^{(i)} = (x^{(i)}_1, x^{(i)}_2, ..., x^{(i)}_d)$)

    - $y^{(i)}$: rótulo de classe pertencente a um conjunto finito de classes $C = \{c_1, c_2, ..., c_k\}$ (label) 

    - $x^o$ : Vetor de features ($x^o = (x_1, x_2, ..., x_d)$), que precisa ser classficado na previsão


**Processamento**

Objetivo é encontrar a classe $\hat{y}$ que maximiza a probabilidade a posteriori $P(y|x^o)$

1. Teorema de Bayes: 

Para cada classe $c_k \in C$, calculamos a probabilidade a posteriori:

$P(c_k|x^o) = \frac{P(x^o|c_k)P(c_k)}{P(x^o)}$

Onde, como já vimos:

- $P(c_k|x^o):$ **Probabilidade a posteriori**
- $P(c_k): **Probabilidade a priori** a classe $c_k$
- $P(x^o| c_k):$ **Verossimilhança (likelihood) da amostra $x^o$ dada a classe $c_k$
- $P(X):$ **Evidência (evidence)**

2. Cálculo da Probabilidade  a Priori $P(c_k):

A priori é estimada a partir da frequência relativa de cada classe no conjunto de treinamento:

$P(c_k) = \frac{\text{Número de amostras da classe } c_k}{\text{Número total de amostras}} = \frac{\sum_{i=1}^{n} I(y^{(i)} = c_k)}{n}$

3. Cálculo da Verossimilhança $P(x^o|c_k)$:

Aqui reside a suposição naive (ingênua) do algoritmo: todas as features são condicionalmente independentes, dada a classe. Matematicamente, isso simplifica o cálculo da verossimilhança:

$P(x^o|c_k) = P(x_1, x_2, ...,x_d|c_k) = \prod^d_{j=1}P(x_j|c_k)$

O cálculo de cada $P(x_j|c_k)$ depende da natureza da feature $x_j$, o que nos leva ao diferentes tipos de modelos Naive Bayes, que veremos!

4. Tomada de Decisão (MAP):

Como o termo P(x^o) é constante para todas as classes, ele pode ser descartado na etapa da maximização. Portanto, a regra de decisão se torna:

$\hat{y} = \arg \max_{c_k \in C} P(c_k) \prod^d_{j=1}P(x_j|c_k) $

- O rótulo da classe predita $\hat{y} \in$ C para $x^o$


**II. Tipos de Modelos de Processamento da Verossimilhança**
Praticamente qualquer diferença entre os tipos de Naive Bayes está emm como eles modelam a distribuição $P(x_j|c_k)$ para diferentes tipos de dados. Alguns deles são:

1. Gaussian Naive Bayes:

- Uso: Features contínuas que se assume seguir uma distribuição normal (Gaussiana)

- Hipótese: Para cada classe $c_k$, o valor da feature $x_j$ é distribuido segundo uma Gaussiana

- Processamento: Durante o treinamento, o algoritmo calcula a média $\mu_{j,k}$ e a variância $\sigma^2_{j,k}$ da feature $j$ para todas as amostras da classe $c_k$. A verossimilhança é então calculado usando a função de densidade de probabilidade (PDF) da distribuição normal:

$P(x_j|c_k) = \frac{1}{2\pi\sigma^2_{j,k}}\exp\left(-\frac{(x_j-\mu_{j,k})^2}{2\sigma^2_{j,k}}\right) $

Saber mais: [GaussianNB](https://www.geeksforgeeks.org/machine-learning/gaussian-naive-bayes/)

2. Multinomial Naive Bayes:

- Uso: Features discretas que representam contagens ou frequências (clássico para classificação de texto com TF ou TF-IDF).

- Hipótese: As features são geradas a partir de uma distribuição multinomial.

- Processamento: A verossimilhança $P(x_j|c_k)$ é a frequência relativa da feature $x_j$ (e.g., uma palavra) dentro dos documentos da classe de $c_k$. Para evitar o problema de frequência zero, utiliza-se o Laplace Smoothing:

$P(x_j|c_k) = \frac{N_{i,j} + \alpha}{N_k + \alpha d}$

Onde:

- $N_{j,k}:$ contagem da feature $x_j$ na classe $c_k$
- $N_k:$ contagem total de todas as features na classe $c_k$
- $d$: o número total de features únicas (tamanho do vocabulário).
- $\alpha:$ é o parâmetro de suavização ($\alpha = 1$ para Laplace smoothing).

Saber mais: [MultionomiaNB](https://www.geeksforgeeks.org/machine-learning/multinomial-naive-bayes/)

3. Bernoulli Naive Bayes:

- Uso: Features binárias (Bernoulli) (e.g., presença ou ausência de uma palavra em um documento)

- Hipótese: As features são varáveis binárias independentes.

- Processamento: A verossimilhamça é calculada para a presença ($x_j = 1$) da feature na classe $c_k$. O cálculo da probabilidade total da amostra considera tanto as features presentes quanto as ausentes: 

$P(x_j|c_k) = P(j|c_k)^{x_j}(1-P(j|c_k))^{(1-x_j)}$

Onde $P(j|c_k)$ é a probabilidade da feature $j$ ocorrer na classe $c_k$.

Saber mais: [BernoulliNB](https://www.geeksforgeeks.org/machine-learning/bernoulli-naive-bayes/)



### 3.1.4 Desafio: "From Scratch"



Vamos implementar o algoritmo naive bayes do zero:

In [ ]:
class NaiveBayesClassifier:
    def __init__(self, alpha=1.0):
        """
        Inicializa o classificador Naive Bayes 

        Parâmetros:
        alpha: parâmetro de suavização de Laplace (default=1.0)

        alpha=1.0 é "add-one smoothing"
        alpha=0.0  significa sem suavização
        """
        pass
    
    def fit(self, X, y):
        """
        Treina o classificador Naive Bayes

        Parâmetros:
        X: lista de listas de palavras (features)
        y: lista de labels (classes)
        """
        pass

    def _preprocess(self, text):
        """
        Pré-processa o texto de entrada

        Parâmetros:
        text: string de texto a ser processada

        Retorna:
        string: texto processado
        """
        pass
        
    def _calculate_priors(self):
        """
        Calcula as probabilidades a priori P(class)

        Fórmula: P(class) = numbers of class's messages / total messages
        """
        pass

    def _calculate_likelihoods(self):
        """
        Calcula as verossimilhanças P(word|class) com suavização de Laplace

        Fórmula: P(word|class) = (count(word in class) + alpha) / (total words in class + alpha * V)
        Onde V é o tamanho do vocabulário (número de palavras únicas)
        """
        pass

    def _calculate_log_probabilities(self, text, class_label):
        """
        Calcula a probabilidade logarítmica de uma mensagem pertencer a uma classe

        Parâmetros:
        text: lista de palavras da mensagem
        class_label: label da classe (spam ou ham)

        Retorna:
        float: probabilidade logarítmica
        """
        pass

    def predict(self, text):
        """
        Classifica uma nova mensagem

        Parâmetros:
        text: string de texto da mensagem a ser classificada

        Retorna:
        string: label da classe prevista (spam ou ham)
        """
        pass

    def predict_proba(self, text):
        """
        Calcula as probabilidades de uma mensagem pertencer a cada classe

        Parâmetros:
        text: string de texto da mensagem

        Retorna:
        dict: dicionário com as probabilidades para cada classe
        """
        pass

    def score(self, X, y):
        """
        Avalia a acurácia do classificador no conjunto de dados fornecido

        Parâmetros:
        X: lista de listas de palavras (features)
        y: lista de labels (classes)

        Retorna:
        float: acurácia do classificador
        """
        pass

In [ ]:
# Dados para Teste (Spam e Ham)
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.model_selection import train_test_split

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "uciml/sms-spam-collection-dataset",
  "spam.csv",
  pandas_kwargs={"encoding": "latin", "usecols": [0,1], "names": ["class", "sms"], 'header': 0}
)

X = df['sms'].tolist()
y = df['class'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### 3.1.5 Usando SKlearn



In [42]:
# Carregando o Dataset (denovo)
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.model_selection import train_test_split

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "uciml/sms-spam-collection-dataset",
  "spam.csv",
  pandas_kwargs={"encoding": "latin", "usecols": [0,1], "names": ["class", "sms"], 'header': 0}
)

X = df['sms'].tolist()
y = df['class'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB
from sklearn.metrics import accuracy_score



In [52]:
# =======================================================
# 1. Essencial
# Ideia:
# - Vetorizar
# - Treinar o modelo
# =======================================================

vectorizer = CountVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Vocabulario: {vectorizer.get_feature_names_out()[1000:5000:400]} ...")
print(f"X_train_vec shape: {X_train_vec.shape}")
print(f"X_test_vec shape: {X_test_vec.shape}")

# ---

nb = MultinomialNB()
nb.fit(X_train_vec, y_train)
y_pred = nb.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nAcurácia: {accuracy*100:.2f}%")

Vocabulario: ['aphexåõs' 'blonde' 'chinese' 'decent' 'enemies' 'free' 'hellogorgeous'
 'jay' 'lotr' 'msgrcvd18'] ...
X_train_vec shape: (4457, 7735)
X_test_vec shape: (1115, 7735)

Acurácia: 98.39%


In [59]:
# 2. Usando as Variantes do NB

# 2.1. MultinomialNB (dados de contagem)
nb_multi = MultinomialNB()
nb_multi.fit(X_train_vec, y_train)
accuracy_multi = nb_multi.score(X_test_vec, y_test)


# 2.2. BernoulliNB (dados binários)
# Transformar os dados em binários (0 ou 1)
X_train_bin = (X_train_vec > 0).astype(int)
X_test_bin = (X_test_vec > 0).astype(int)

nb_bernoulli = BernoulliNB()
nb_bernoulli.fit(X_train_bin, y_train)
accuracy_bernoulli = nb_bernoulli.score(X_test_bin, y_test)


# 2.3. GaussianNB (dados contínuos)
# Converter os dados esparsos para densos (GaussianNB não aceita sparse)
X_train_dense = X_train_vec.toarray()
X_test_dense = X_test_vec.toarray()

nb_gaussian = GaussianNB()
nb_gaussian.fit(X_train_dense, y_train)
accuracy_gaussian = nb_gaussian.score(X_test_dense, y_test)

print('''
Nota:

1. MultinomialNB: {:.2f}% de acurácia - Melhor para classificação de texto, especialmente com contagem de palavras (e.g. Classificação de emails)
2. BernoulliNB: {:.2f}% de acurácia - Bom para textos curtos com recursos binários (e.g. Classificação de tweets)
3. GaussianNB: {:.2f}% de acurácia - Menos adequado para texto, melhor para dados contínuos (e.g. Classificação de imagens/sinais)
'''.format(accuracy_multi*100, accuracy_bernoulli*100, accuracy_gaussian*100))



Nota:

1. MultinomialNB: 98.39% de acurácia - Melhor para classificação de texto, especialmente com contagem de palavras (e.g. Classificação de emails)
2. BernoulliNB: 97.49% de acurácia - Bom para textos curtos com recursos binários (e.g. Classificação de tweets)
3. GaussianNB: 90.04% de acurácia - Menos adequado para texto, melhor para dados contínuos (e.g. Classificação de imagens/sinais)



In [62]:
# Parâmetros Importantes do MultinomialNB
# - alpha: parâmetro de suavização de Laplace (default=1.0)
#   - alpha=1.0 é "add-one smoothing"
#   - alpha=0.0  significa sem suavização

# fit_prior (default=True): se True, aprende a probabilidade a priori das classes a partir dos dados de treinamento. Se False, assume que todas as classes são igualmente prováveis.

# class_prior (default=None): se fit_prior for False, você pode fornecer suas próprias probabilidades a priori para as classes.
# =======================================================

alphas = [0.001, 0.1, 0.5, 1.0, 2.0, 5.0]
results = []
for alpha in alphas:
    nb = MultinomialNB(alpha=alpha)
    nb.fit(X_train_vec, y_train)
    accuracy = nb.score(X_test_vec, y_test)
    results.append((alpha, accuracy))
    print(f"Alpha: {alpha}, Acurácia: {accuracy*100:.2f}%")


Alpha: 0.001, Acurácia: 98.30%
Alpha: 0.1, Acurácia: 98.39%
Alpha: 0.5, Acurácia: 98.21%
Alpha: 1.0, Acurácia: 98.39%
Alpha: 2.0, Acurácia: 97.85%
Alpha: 5.0, Acurácia: 97.04%


In [65]:
# =======================================================
# COUNTVECTORIZER vs TFIDF VECTORIZER

# CounterVectorizer: conta a frequência das palavras
# TFIDFVectorizer: considera a frequência das palavras e a importância delas (menos frequentes são mais importantes)

# Qual é melhor? Depende!
# =======================================================

vectorizer_count = CountVectorizer()
X_train_count = vectorizer_count.fit_transform(X_train)
X_test_count = vectorizer_count.transform(X_test)

nb_count = MultinomialNB()
nb_count.fit(X_train_count, y_train)
acc_count = nb_count.score(X_test_count, y_test)

vectorizer_tfidf = TfidfVectorizer()
X_train_tfidf = vectorizer_tfidf.fit_transform(X_train)
X_test_tfidf = vectorizer_tfidf.transform(X_test)

nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
acc_tfidf = nb_tfidf.score(X_test_tfidf, y_test)
print(f"CountVectorizer Acurácia: {acc_count*100:.2f}%")
print(f"TFIDFVectorizer Acurácia: {acc_tfidf*100:.2f}%")

print('''
Nota:
1. CountVectorizer: {:.2f}% de acurácia - Simples e eficaz para muitos casos, normalmente funciona melhor para NB
2. TFIDFVectorizer: {:.2f}% de acurácia - Útil quando a importância das palavras varia muito
'''.format(acc_count*100, acc_tfidf*100))

CountVectorizer Acurácia: 98.39%
TFIDFVectorizer Acurácia: 96.23%

Nota:
1. CountVectorizer: 98.39% de acurácia - Simples e eficaz para muitos casos, normalmente funciona melhor para NB
2. TFIDFVectorizer: 96.23% de acurácia - Útil quando a importância das palavras varia muito



### 3.1.6 Conclusões e Considerações

**Suposição de Independência Condicional:** Embora MUITO raramente seja verdadeira em cenários reais (e.g., as palavras "São" e "Paulo" não são independentes), o Naive Bayes frequentemente apresenta uma performance muito boa. Isso acontece porque a classificação não exige uma estimativa de probabilidade perfeitamente calibrada, mas apenas que a classe correta tenha a maior pontuação a posteriori

**Zero-Frequency Problem**: Se uma featurer de uma nova amostra não foi vista durante o treinamento para uma determinada classe, sua probabilidade $P(x_j|c_K)$ seria 0. Devido ao produtório, isso anularia a posteriori para aquela classe. O smoothing é a técnica que você deve usar para mitigar esse problema.

**Estabilidade Numérica (Log-Probabilities):** A multiplicação de muitas probabilidade pode resultar em valores extremamentes pequenos, levando a problemas de underflow (arrendondar para 0). Por isso normalmente, trabalhamos com a soma dos logaritmos das probabilidades, transformando o produtório em um somatório. A regra de decisão se torna:

$\hat{y} = \arg \max_{c_k \in C} \left(\log{(P(c_k)+\sum^d_{j=1}\log{(P(x_j|c_k))}} \right) $

**Aplicações do Naive Bayes Classifier**:

- Filtragem de e-mail de spam

- Classificação de texto (e.g., análise de sentimento e categorização de documentos)

- Diagnóstico médico: probabilidade de uma doença, baseado nos sintomas

- Pontuação de crédito: avaliação de credibilidade para aprovação de empréstimos

- Previsão do tempo: classificação de condições meteorológicas com base em vários fatores

